# 20a · CPU and Metal inference tuning

## The engineering question
When does Metal batching beat CPU inference, and does the representation remain suitable for the same retrieval task?

**Learning outcomes:** separate model loading from warm inference; control batch size, CPU threads, and input length; synchronize GPU work; compare numerical drift and retrieval quality alongside throughput.

Both devices use the same pinned MiniLM weights and float32 output contract. We time tokenization, inference, normalization, and transfer back to NumPy together. MPS is explicitly synchronized before timing ends. The workload uses 48 repeated teaching sentences; it is a hardware diagnostic, not representative traffic. First-use timing includes model loading and initialization but does not imply a cold filesystem cache.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vector_masterclass.lesson import LessonRun
from vector_masterclass.config import root

plt.rcParams.update(
    {"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False}
)
lesson = LessonRun("20a", "B", docker=False)
print("Kernel:", sys.executable)
print("Workspace:", root())

In [ ]:
import time
import torch
from vector_masterclass.models import embed, dense_model, release_models
from vector_masterclass.data import documents, queries
from vector_masterclass.retrieval import exact_search
from vector_masterclass.metrics import ranking_metrics

docs = documents(tenant="public")
ids = [d.key for d in docs]
dev = queries("development")
rows, quality, outputs, loads = [], [], {}, []
for device in ["cpu", "mps"]:
    started = time.perf_counter()
    model = dense_model(device=device)
    embed(["Atlas warranty"], device=device)
    loads.append({"device": device, "first_load_and_encode_seconds": time.perf_counter() - started})
    for threads in [1, 4] if device == "cpu" else [4]:
        torch.set_num_threads(threads)
        for repetitions in [1, 12]:
            texts = [
                ("Atlas device warranty lasts two years. " * repetitions).strip() for _ in range(48)
            ]
            token_count = len(model.tokenizer.encode(texts[0]))
            for batch in [1, 16, 48]:
                embed(texts[:batch], device=device, batch_size=batch)
                samples = []
                for repeat in range(3):
                    started = time.perf_counter()
                    vectors = embed(texts, device=device, batch_size=batch)
                    samples.append(time.perf_counter() - started)
                rows.append(
                    {
                        "device": device,
                        "cpu_threads": threads,
                        "input_tokens": token_count,
                        "batch_size": batch,
                        "median_seconds": np.median(samples),
                        "documents_per_second": 48 / np.median(samples),
                        "samples_seconds": samples,
                    }
                )
    dv = embed([d.text for d in docs], device=device)
    qv = embed([q.text for q in dev], device=device, role="query")
    outputs[device] = dv
    for q, v in zip(dev, qv):
        ranked = [i for i, s in exact_search(v, dv, ids, k=5)]
        quality.append({"device": device, "query": q.id, **ranking_metrics(ranked, q.relevance, 5)})
    release_models()
torch.set_num_threads(4)
results = lesson.table(pd.DataFrame(rows))
display(results.drop(columns="samples_seconds"))
display(lesson.table(pd.DataFrame(loads), "first-use"))
quality = lesson.table(pd.DataFrame(quality), "quality")
display(quality.groupby("device")[["recall", "mrr", "ndcg"]].mean())
drift = float(np.max(np.abs(outputs["cpu"] - outputs["mps"])))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (tokens, group) in zip(axes, results.groupby("input_tokens")):
    for (device, threads), part in group.groupby(["device", "cpu_threads"]):
        ax.plot(
            part.batch_size, part.documents_per_second, "o-", label=f"{device}; threads={threads}"
        )
    ax.set(title=f"{tokens} tokens per input", xlabel="Batch size", ylabel="Documents / second")
axes[1].legend(loc="upper left", bbox_to_anchor=(1, 1))
fig.tight_layout()
lesson.figure(fig)
plt.show()
print("Maximum absolute CPU/MPS vector difference:", drift)

## Interpretation and exercise
Small requests can be dominated by dispatch/transfer costs; larger batches can improve GPU utilization while increasing individual waiting time and unified-memory pressure. CPU oversubscription competes with Qdrant and the notebook kernel. Changing maximum sequence length changes the representation of long documents, so it is a quality decision as well as a speed knob. This experiment varies actual input length without truncating different semantic content for its quality comparison.

**Exercise:** choose a serving batch size for interactive queries and a separate ingestion batch size. Explain the difference using these measurements. Add a long document whose relevant sentence is beyond the configured 256-token boundary and inspect the resulting retrieval loss before shortening any limit.

A production benchmark needs arrival rates, batching delay, concurrent storage traffic, thermal behavior, and memory pressure. Three warm repetitions are descriptive. MPS fallback is disabled: unsupported operations must fail visibly. [PyTorch MPS synchronization](https://docs.pytorch.org/docs/stable/generated/torch.mps.synchronize.html).

In [ ]:
manifest = lesson.finish(
    {
        "eighteen_configurations": len(results) == 18,
        "same_weights_both_devices": set(quality.device) == {"cpu", "mps"},
        "finite_outputs": all(np.isfinite(v).all() for v in outputs.values()),
        "bounded_numerical_drift": drift < 1e-4,
    },
    observations={
        "max_absolute_difference": drift,
        "timing_scope": "warm end-to-end encoder including tokenizer and synchronized device transfers",
        "first_use_is_not_cold_disk": True,
        "quality_split": "development",
    },
)

## References and next steps
Consult the [Qdrant documentation](https://qdrant.tech/documentation/),
the [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
and [uv's Jupyter guide](https://docs.astral.sh/uv/guides/integration/jupyter/).
The exact pinned configuration is in `configs/models.json`; worked exercises are in `docs/SOLUTIONS.md`.
This notebook's measurements apply to the recorded local run, not a production deployment.